In [ ]:
# Import libraries
import pandas as pd
import numpy as np

In [ ]:
# Load datasets
vehicle_insurance_df = pd.read_csv('motor_data14-2018.csv')

# Select relevant columns and rename them for clarity
selected_columns = {
    "SEX": "sex",
    "INSR_BEGIN": "insurance_begin",
    "INSR_END": "insurance_end",
    "EFFECTIVE_YR": "effective_year",
    "INSR_TYPE": "insurance_type",
    "INSURED_VALUE": "insured_value",
    "PREMIUM": "premium",
    "OBJECT_ID": "object_id",
    "PROD_YEAR": "production_year",
    "SEATS_NUM": "seats_number",
    "CARRYING_CAPACITY": "carrying_capacity",
    "TYPE_VEHICLE": "vehicle_type",
    "CCM_TON": "ccm_ton",
    "MAKE": "manufacturer",
    "USAGE": "usage",
    "CLAIM_PAID": "claim_paid",
}

vehicle_insurance_df = vehicle_insurance_df[list(selected_columns.keys())].rename(columns=selected_columns)

# Replace sex values
vehicle_insurance_df["sex"] = vehicle_insurance_df["sex"].replace({0: "legal entity", 1: "male", 2: "female"})

# Replace insurance_type values
vehicle_insurance_df["insurance_type"] = vehicle_insurance_df["insurance_type"].replace({1201: "private", 1202: "commercial", 1204: "motor trade road risk"})

# Display data info
print(vehicle_insurance_df.info())
print(vehicle_insurance_df.head())

In [ ]:
# Drop duplicates
vehicle_insurance_df.drop_duplicates(inplace=True)

# Check for missing values
missing_values = vehicle_insurance_df.isnull().sum()
print("Missing values in each column:")
print(missing_values)

# Drop null values for premium, production_year, and manufacturer
vehicle_insurance_df.dropna(subset=["premium", "production_year", "manufacturer"], inplace=True)

# Convert production_year to an integer type
vehicle_insurance_df["production_year"] = vehicle_insurance_df["production_year"].astype(int)

# Check for missing values after dropping
missing_values_after = vehicle_insurance_df.isnull().sum()
print("Missing values after dropping:")
print(missing_values_after)
vehicle_insurance_df.dropna(subset=["premium", "production_year", "manufacturer"], inplace=True)

# Convert categorical variables to proper types
vehicle_insurance_df["sex"] = vehicle_insurance_df["sex"].astype("category")
vehicle_insurance_df["insurance_type"] = vehicle_insurance_df["insurance_type"].astype("category")
vehicle_insurance_df["vehicle_type"] = vehicle_insurance_df["vehicle_type"].astype("category")
vehicle_insurance_df["manufacturer"] = vehicle_insurance_df["manufacturer"].astype("category")
vehicle_insurance_df["usage"] = vehicle_insurance_df["usage"].astype("category")

# Check for zero values in each selected column
zero_values = (vehicle_insurance_df == 0).sum()
print("Zero values in each column:")
print(zero_values)

# Drop zero values in the premium column
vehicle_insurance_df = vehicle_insurance_df[vehicle_insurance_df["premium"] > 0]
vehicle_insurance_df.dropna(subset=["premium", "production_year", "manufacturer"], inplace=True)

# Replace 0s with median in the insured value column
median_insured_value_df = vehicle_insurance_df[vehicle_insurance_df['insured_value'] > 0].groupby(['manufacturer', 'vehicle_type', 'production_year'],
                                                                                                  dropna=False)['insured_value'].median().dropna().reset_index()
median_insured_value_dict = median_insured_value_df.set_index(['manufacturer', 'vehicle_type', 'production_year'])['insured_value'].to_dict()

def find_closest_year_match(manufacturer, vehicle_type, production_year):
    """Find the closest production year match if an exact match is not available, within a maximum difference of 5 years."""
    subset = median_insured_value_df[(median_insured_value_df['manufacturer'] == manufacturer) &
                                     (median_insured_value_df['vehicle_type'] == vehicle_type)]
    if subset.empty:
        return np.nan
    subset = subset[(subset['production_year'] - production_year).abs() <= 5]
    if subset.empty:
        return np.nan
    closest_year = subset.loc[(subset['production_year'] - production_year).abs().idxmin()]
    return closest_year['insured_value'] if not closest_year.empty else np.nan

def replace_zero_with_median(row):
    """Replace zero insured_value with the median value based on relaxed conditions."""
    key = (row['manufacturer'], row['vehicle_type'], row['production_year'])
    median_value = median_insured_value_dict.get(key, np.nan)
    
    if pd.notna(median_value) and median_value > 0:
        return median_value
    else:
        return find_closest_year_match(row['manufacturer'], row['vehicle_type'], row['production_year'])

# Apply replacements efficiently using vectorized approach
mask = vehicle_insurance_df['insured_value'] == 0
vehicle_insurance_df.loc[mask, 'insured_value'] = vehicle_insurance_df.loc[mask].apply(replace_zero_with_median, axis=1)

# Check for zero values after replacing
zero_values = (vehicle_insurance_df == 0).sum()
print("Zero values in each column after replacing:")
print(zero_values)

# Drop remaining zero-value rows in insured_value
vehicle_insurance_df = vehicle_insurance_df[vehicle_insurance_df['insured_value'] > 0]

# Check for zero values after dropping
zero_values = (vehicle_insurance_df == 0).sum()
print("Zero values in each column after dropping:")
print(zero_values)

In [ ]:
# Convert insurance_end and insurance_begin to date format
vehicle_insurance_df["insurance_begin"] = pd.to_datetime(vehicle_insurance_df["insurance_begin"], errors='coerce')
vehicle_insurance_df["insurance_end"] = pd.to_datetime(vehicle_insurance_df["insurance_end"], errors='coerce')

# Calculate insurance_duration in days
vehicle_insurance_df["insurance_duration"] = (vehicle_insurance_df["insurance_end"] - vehicle_insurance_df["insurance_begin"]).dt.days

# Create insurance_claim column
vehicle_insurance_df["insurance_claim"] = vehicle_insurance_df["claim_paid"].apply(lambda x: "claimed" if pd.notna(x) else "no claim")

# Display cleaned data info
print(vehicle_insurance_df.info())
print(vehicle_insurance_df.head())

# Save the updated DataFrame (optional)
# vehicle_insurance_df.to_csv("/mnt/data/updated_vehicle_insurance.csv", index=False)

In [ ]:
vehicle_insurance_df[['insured_value', 'premium', 'production_year', 'claim_paid']].describe().apply(lambda x: x.map('{:,.0f}'.format))

In [ ]:
import plotly.graph_objects as go
import plotly.express as px

In [ ]:
######## pie chart gender
sex_counts = vehicle_insurance_df['sex'].value_counts()
fig = go.Figure(data=[go.Pie(labels=sex_counts.index, values=sex_counts.values)])
fig.update_layout(
    title="Distribution of Sex",
    height=400, 
    width=500, 
    showlegend=True
)
fig.show()

######## pie chart insurance_type
insurance_type_counts = vehicle_insurance_df['insurance_type'].value_counts()
fig = go.Figure(data=[go.Pie(labels=insurance_type_counts.index, values=insurance_type_counts.values)])
fig.update_layout(
    title="Distribution of Insurance Type",
    height=400, 
    width=500, 
    showlegend=True
)
fig.show()

######## bar chart vehicle_type
vehicle_type_counts = vehicle_insurance_df['vehicle_type'].value_counts().sort_values(ascending=True)
fig = go.Figure(data=[go.Bar(x=vehicle_type_counts.values, y=vehicle_type_counts.index,orientation='h',marker=dict(color='darkgreen'))])
fig.update_layout(
    title="Distribution of Vehicle Type",
    height=500, 
    width=700,
    plot_bgcolor='rgba(0,0,0,0)',
    showlegend=False
)
fig.show()

######## bar chart usage
usage_counts = vehicle_insurance_df['usage'].value_counts().sort_values(ascending=True)
fig = go.Figure(data=[go.Bar(x=usage_counts.values, y=usage_counts.index,orientation='h',marker=dict(color='darkgreen'))])
fig.update_layout(
    title="Distribution of Usage",
    height=500, 
    width=700,
    plot_bgcolor='rgba(0,0,0,0)',
    showlegend=False
)

fig.show()

In [ ]:
# Check the distribution of production_year, insured_value, and premium
num_cols = ["production_year", "insured_value", "premium"]
figs = []
for col in num_cols:
    fig = px.histogram(vehicle_insurance_df, x=col, nbins=50, marginal="box", 
                       opacity=0.9, color_discrete_sequence=['darkcyan'])
    fig.update_traces(marker_line_color='black', marker_line_width=0.5)
    fig.update_layout(
        height=400, 
        width=550,
        plot_bgcolor='rgba(0,0,0,0)',
        showlegend=False
    )
    figs.append(fig)

for fig in figs:
    fig.show()



In [ ]:
# Take log of insured_value and premium to check distributions
vehicle_insurance_df["log_insured_value"] = vehicle_insurance_df["insured_value"].apply(lambda x: np.log(x) if x > 0 else 0)
vehicle_insurance_df["log_premium"] = vehicle_insurance_df["premium"].apply(lambda x: np.log(x) if x > 0 else 0)

num_cols = ["log_insured_value", "log_premium"]
figs = []
for col in num_cols:
    fig = px.histogram(vehicle_insurance_df, x=col, nbins=50, marginal="box", 
                       color_discrete_sequence=['darkcyan'])
    fig.update_traces(marker_line_color='black', marker_line_width=0.5)
    fig.update_layout(
        height=400, 
        width=550,
        plot_bgcolor='rgba(0,0,0,0)',
        showlegend=False
    )
    figs.append(fig)

for fig in figs:
    fig.show()

In [ ]:
average_premium_per_year = vehicle_insurance_df.groupby('production_year')['premium'].mean()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=average_premium_per_year.index,
    y=average_premium_per_year.values,
    mode='lines',
    line=dict(color='darkblue')
))

fig.update_layout(
    title='Average Premium vs Production Year',
    xaxis_title='Production Year',
    yaxis_title='Average Premium Amount',
    plot_bgcolor='rgba(0,0,0,0)',
    xaxis=dict(showgrid=True, gridcolor='lightgray'), 
    yaxis=dict(showgrid=True, gridcolor='lightgray'),
    height=400, 
    width=550
)
fig.show()

In [ ]:
#vehicle_insurance_df['insured_value_rounded'] = vehicle_insurance_df['insured_value'].round(-2)
average_premium_per_year = vehicle_insurance_df.groupby('log_insured_value')['premium'].mean()
average_premium_per_year = average_premium_per_year.rolling(window=3, min_periods=1).mean()
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=average_premium_per_year.index,
    y=average_premium_per_year.values,
    mode='lines',
    line=dict(color='darkblue')
))

fig.update_layout(
    title='Average Premium vs Insured Value (log)',
    xaxis_title='Insured Value (log)',
    yaxis_title='Average Premium Amount',
    plot_bgcolor='rgba(0,0,0,0)',
    xaxis=dict(showgrid=True, gridcolor='lightgray'), 
    yaxis=dict(showgrid=True, gridcolor='lightgray'),
    height=400, 
    width=550
)
fig.show()

In [ ]:
# Plot average premium vs production year
average_premium_per_year = vehicle_insurance_df.groupby('production_year')['premium'].mean()
average_premium_private = vehicle_insurance_df[vehicle_insurance_df['insurance_type'] == 'Private'].groupby('production_year')['premium'].mean()
average_premium_motor_trade = vehicle_insurance_df[vehicle_insurance_df['insurance_type'] == 'Motor Trade Road Risk'].groupby('production_year')['premium'].mean()

average_premium_per_year
average_premium_private
average_premium_motor_trade

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=average_premium_per_year.index,
    y=average_premium_per_year.values,
    mode='lines',
    line=dict(color='darkblue'),
    name='Overall Average Premium'
))
fig.add_trace(go.Scatter(
    x=average_premium_private.index,
    y=average_premium_private.values,
    mode='lines',
    line=dict(color='red', dash='dash'),
    name='Private Insurance Average Premium'
))
fig.add_trace(go.Scatter(
    x=average_premium_motor_trade.index,
    y=average_premium_motor_trade.values,
    mode='lines',
    line=dict(color='green', dash='dot'),
    name='Motor Trade Road Risk Average Premium'
))

fig.update_layout(
    title='Average Premium vs Production Year',
    xaxis_title='Production Year',
    yaxis_title='Average Premium Amount',
    plot_bgcolor='rgba(0,0,0,0)',
    xaxis=dict(showgrid=True, gridcolor='lightgray'), 
    yaxis=dict(showgrid=True, gridcolor='lightgray'),
    height=400, 
    width=550,
    legend=dict(
        title='Legend',
        x=0.02,  # Positioning of the legend
        y=0.98,
        bgcolor='rgba(255,255,255,0.5)',
        bordercolor='black',
        borderwidth=1
    )
)
fig.show()